In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from scipy.stats import pearsonr

In [3]:
train = pd.read_parquet('../data/cleanedTrain.parquet')

In [4]:
feature_cols = [col for col in train.columns if col not in ['timestamp', 'label']]

X = train[feature_cols]
y = train['label']

In [5]:
# Replace inf/-inf with NaN
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check how many NaNs remain
print(X.isnull().sum().sum())

# Simple fix: fill NaNs with median (safer for financial data)
X.fillna(X.median(), inplace=True)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13152\2556922024.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace([np.inf, -np.inf], np.nan, inplace=True)


11043627


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13152\2556922024.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(X.median(), inplace=True)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
model = xgb.XGBRegressor(
    n_estimator=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'
)

In [8]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)]
)

y_pred = model.predict(X_val)

c:\Users\LENOVO\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [13:04:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimator" } are not used.

  warnings.warn(smsg, UserWarning)


[0]	validation_0-rmse:1.01149
[1]	validation_0-rmse:1.00645
[2]	validation_0-rmse:1.00222
[3]	validation_0-rmse:0.99801
[4]	validation_0-rmse:0.99433
[5]	validation_0-rmse:0.99047
[6]	validation_0-rmse:0.98749
[7]	validation_0-rmse:0.98356
[8]	validation_0-rmse:0.98008
[9]	validation_0-rmse:0.97612
[10]	validation_0-rmse:0.97379
[11]	validation_0-rmse:0.97041
[12]	validation_0-rmse:0.96730
[13]	validation_0-rmse:0.96326
[14]	validation_0-rmse:0.95960
[15]	validation_0-rmse:0.95590
[16]	validation_0-rmse:0.95269
[17]	validation_0-rmse:0.95029
[18]	validation_0-rmse:0.94687
[19]	validation_0-rmse:0.94416
[20]	validation_0-rmse:0.94187
[21]	validation_0-rmse:0.93906
[22]	validation_0-rmse:0.93644
[23]	validation_0-rmse:0.93395
[24]	validation_0-rmse:0.93110
[25]	validation_0-rmse:0.92723
[26]	validation_0-rmse:0.92470
[27]	validation_0-rmse:0.92126
[28]	validation_0-rmse:0.91950
[29]	validation_0-rmse:0.91613
[30]	validation_0-rmse:0.91344
[31]	validation_0-rmse:0.91073
[32]	validation_0-

In [9]:
RMSE = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(f"RMSE: {RMSE:.5f}")
print(f"R2 score: {r2:.5f}")

corr, _ = pearsonr(y_val, y_pred)
print(f"Validation Pearson correlation: {corr:.5f}")

RMSE: 0.59914
R2 score: 0.41999
Validation Pearson correlation: 0.73863


In [10]:
train['label'].std()

1.0099135624525382

In [11]:
joblib.dump(model, '../results/XGB/xgb_first_model.joblib')

['../results/XGB/xgb_first_model.joblib']